# Multi-head latent attention with decoupled RoPE

Multi-head latent attention (MLA) caches one small **latent** vector per token instead of per-head keys and values. Keys and values for every head are re-expanded from that latent. RoPE does not commute with the up-projection, so position information travels in a separate, small key that all heads share.

Write `mla(x, W_dkv, W_uk, W_uv, W_kr, W_q, W_o, n_heads, d_rope, base=10000.0)` in PyTorch: causal self-attention over a full sequence (prefill), returning the output and the two tensors an engine would cache.

## Shapes (all float64; projections are $XW$, no biases)
- `x`: `(B, T, D)`. $H$ = `n_heads`, $d_r$ = `d_rope` (even), $d_c$ = latent width, $d_h$ = per-head content width.
- `W_dkv`: `(D, d_c)`, `W_uk`: `(d_c, H d_h)`, `W_uv`: `(d_c, H d_h)`, `W_kr`: `(D, d_r)`, `W_q`: `(D, H (d_h + d_r))`, `W_o`: `(H d_h, D)`.

## Computation
1. Latent: $c = x W_{dkv}$, `(B, T, d_c)`. Content keys $k^C = c\,W_{uk}$ and values $v = c\,W_{uv}$, split into heads of width $d_h$ (head $h$ = columns $h d_h : (h+1) d_h$).
2. Shared position key: $k^R = \mathrm{RoPE}(x W_{kr})$, `(B, T, d_r)`, one for all heads.
3. Queries: reshape $x W_q$ to `(B, T, H, d_h + d_r)`. In each head the first $d_h$ features are $q^C$, the last $d_r$ are $q^R$, and $q^R \leftarrow \mathrm{RoPE}(q^R)$.
4. Scores for head $h$: $\;s_{ij} = \bigl(q^C_{i} \cdot k^C_{j} + q^R_{i} \cdot k^R_{j}\bigr) / \sqrt{d_h + d_r}$, causal ($j \le i$), softmax over $j$, output $\sum_j a_{ij} v_j$.
5. Concatenate heads in order to `(B, T, H d_h)` and multiply by $W_o$.

**RoPE** here uses the "half" layout: for a vector $u$ of width $d_r$ at position $m$ (tokens are at positions $0, \dots, T-1$), pair $i$ is features $(i, i + d_r/2)$, rotated by angle $m\,\theta_i$ with $\theta_i = \text{base}^{-2i/d_r}$:
$$(u_i, u_{i+d_r/2}) \mapsto (u_i \cos m\theta_i - u_{i+d_r/2} \sin m\theta_i,\;\; u_i \sin m\theta_i + u_{i+d_r/2} \cos m\theta_i).$$

## Output
A tuple `(y, c, k_rope)`: `y` is `(B, T, D)`, `c` is `(B, T, d_c)`, `k_rope` is `(B, T, d_r)` (after RoPE).

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def mla(x, W_dkv, W_uk, W_uv, W_kr, W_q, W_o, n_heads, d_rope, base=10000.0):
    """Return (y (B,T,D), latent cache c (B,T,dc), rotated shared key k_rope (B,T,d_rope))."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[1.0, 0.0], [0.0, 1.0]]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0, 1.0, 0.0], [0.0, 1.0, 0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, 1, 2], "e": {"__tup__": [{"__nd__": [[[1.0, 0.0], [0.19454630747957052, 0.8054536925204294]]], "dtype": "float64"}, {"__nd__": [[[1.0, 0.0], [0.0, 1.0]]], "dtype": "float64"}, {"__nd__": [[[1.0, 0.0], [-0.8414709848078965, 0.5403023058681398]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0, 1.0, 0.0], [0.0, 1.0, 0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, 1, 2], "e": {"__tup__": [{"__nd__": [[[1.0, 0.0], [0.19454630747957052, 0.8054536925204294], [0.7146299666962297, 0.9262870922033895]]], "dtype": "float64"}, {"__nd__": [[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]], "dtype": "float64"}, {"__nd__": [[[1.0, 0.0], [-0.8414709848078965, 0.5403023058681398], [-1.325444263372824, 0.4931505902785393]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[0.30471707975443135, -1.0399841062404955, 0.7504511958064572, 0.9405647163912139, -1.9510351886538364, -1.302179506862318, 0.12784040316728537, -0.3162425923435822], [-0.016801157504288795, -0.85304392757358, 0.8793979748628286, 0.7777919354289483, 0.06603069756121605, 1.1272412069680329, 0.4675093422520456, -0.8592924628832382], [0.36875078408249884, -0.9588826008289989, 0.8784503013072725, -0.049925910986252896, -0.18486236354526056, -0.6809295444039414, 1.2225413386740303, -0.15452948206880215], [-0.4283278221631072, -0.3521335504882296, 0.5323091855533487, 0.36544406436407834, 0.4127326115959884, 0.43082100300788273, 2.1416476008704612, -0.4064150163846156]], [[-0.5122427290715373, -0.8137727282478777, 0.6159794225754956, 1.1289722927208916, -0.11394745765487507, -0.840156476962528, -0.8244812156912396, 0.6505927878247011], [0.7432541712034423, 0.543154268305195, -0.6655097072886943, 0.23216132306671977, 0.11668580914072822, 0.21868859672901295, 0.8714287779481898, 0.22359554877468227], [0.6789135630718949, 0.06757906948889146, 0.28911939868998415, 0.6312882258385404, -1.4571558198556664, -0.31967121635730134, -0.4703726542927955, -0.6388778482433419], [-0.27514225122668373, 1.4949413112343959, -0.8658311156932432, 0.9682783545914808, -1.6828697716158048, -0.33488502998577485, 0.1627530651050056, 0.5862223313592781]]], "dtype": "float64"}, {"__nd__": [[0.21336797393785648, 0.23800417055997755, -0.10461752167453127, -0.13870553779937014], [0.25739276437714614, -0.057391297464484466, -0.3827058970013765, -0.33998616420104416], [-0.2758356858004834, 0.1491482232161292, 0.042727720821169574, 0.20714560622033046], [-0.12817579390096026, 0.04756190732301427, 0.18767711819020103, -0.09280396191607151], [0.13703257126722343, -0.19857778231999537, -0.10891615396952153, -0.11452136819949872], [-0.3587518936767119, 0.14609174423567453, -0.14082070206081715, 0.0037482356183062286], [0.14422399767177269, 0.13395935280898322, 0.19961553269183588, -0.029545645352827082], [-0.12698949361324613, -0.023915463271919714, -0.506200330187409, -0.43413374172692615]], "dtype": "float64"}, {"__nd__": [[-0.3968098837063207, -0.2991740482804445, 0.11993226801703097, -0.2716437166080182, -0.1134487662118169, 0.3897684893358196], [-0.10687919131842778, 0.22125467054012596, -0.2800853040029631, -0.061631267360289003, -0.2850066164731744, -0.10170992277016874], [0.25209244123721863, -0.5181961269577046, 0.1303270930637572, 0.07132068069968336, -0.17824498670903832, -0.43381735631653634], [0.021638852314160854, -0.15884781271914072, 0.06980286340641119, 0.006555643657032864, 0.4805336673962746, -0.07180668824190728]], "dtype": "float64"}, {"__nd__": [[-0.30704924778655945, 0.053782690486894845, 0.06599900519152954, 0.40775627257213093, 0.25053337377437357, 0.1070613177448528], [0.4389908673658685, -0.3566289162968553, -0.19192545982492432, -0.27797278242165746, -0.11694294094673038, -0.41300584426689263], [0.19054528404432128, -0.06666680912963201, -0.44124188835079736, -0.3046737243622625, 0.09405415423505858, 0.25143797036831433], [0.599019267507536, 0.8741587398021988, 0.12432282998279891, -0.2968614360095592, -0.6396138842193927, 0.08031343870315073]], "dtype": "float64"}, {"__nd__": [[-0.24388232859309777, -0.12460717805390559], [-0.18362903971794242, -0.04223726592491558], [0.31979406923629305, 0.04711457023360339], [-0.04759045111160649, -0.3106961258477435], [-0.5024048834113071, -0.14589237272199926], [-0.016134765245496147, 0.5303789740739648], [0.03908235644186575, 0.29482185330691724], [-0.1497886795617456, -0.3554831299251074]], "dtype": "float64"}, {"__nd__": [[-0.28953502866971154, -0.21756781936072597, 0.6385409197305493, -0.24641600376731582, 0.25154676112090346, -0.2708781534261079, 0.2794719038622732, 0.11548528983175894, -0.04699136929742712, -0.012228757840630208], [-0.1964363086288171, 0.13382166044462415, -0.136495044102234, -0.36768172913017444, -0.38338127229588576, 0.05177637531663584, 0.47372737692313044, 0.04799748407203147, -0.035591497832964766, 0.08574784188076286], [0.39180052251204744, 0.0658147504091569, -0.12327816925012114, 0.3318866130179666, 0.12862693153848404, 0.4607267975987976, 0.05497033116657184, -0.36734070951615005, -0.41044775977369946, 0.4952783796693748], [0.5170997162349891, -0.053855763984780196, -0.11495619634079632, 0.43843328767266065, -0.3321137046130464, -0.2684181056867479, 0.1929980384067133, -0.11838153685787688, -0.0015365600160213888, -0.049032869557353774], [0.10127236463968006, 0.422244558394115, 0.027175472070523662, 0.19318163798305735, -0.6150516303093068, -0.014615520579035384, -0.2529690810878613, -0.3656439181270884, -0.2634457100786252, -0.10023703221024362], [0.2747707627068039, -0.3979178153218692, 0.009189447778325234, -0.14525083000007355, -0.09830192830858825, 0.30082734759138124, 0.1614346311011778, 0.4012194322328231, -0.04635170377497014, -0.2087827835012109], [-0.06715764506414985, 0.07274903738136647, 0.05297200753611331, -0.32531642167000996, 0.027146934488362265, 0.06846849904167154, 0.7552422112601761, 0.5630533833845011, -0.255973005167646, -0.08621500846475283], [-0.43903260055110094, -0.17721210418904593, 0.09468150107710215, 0.36175608626647004, -0.21872515132308254, -0.19624393202033896, -0.6441867089215965, -0.0487997761634723, -0.3187243235578689, -0.1588318282098221]], "dtype": "float64"}, {"__nd__": [[-0.26305823345027646, -0.028278766275767094, -0.5273185174069893, -0.44011357361729714, 0.6387741336084894, -0.38622677438220926, -0.3290356735363919, 0.5510740584963942], [0.8715201507721221, -0.3514699886476025, -0.11047468703640916, 0.10246666528215286, 0.5186092933216777, -0.29605712352847124, -0.07358335378263292, 0.2332012728185232], [0.1304298223399856, -0.11284682136902977, -0.040146889354812344, -0.4124687425109945, -0.07145212319239956, -0.07991624700268653, 0.06965096688787678, -0.16659816564570482], [0.1414615567635417, 0.30381474534594854, 0.04662879830053981, 0.10552692251976103, 0.01594660427336021, 2.531792742425413e-05, -0.21646741006285422, 0.0949482785019924], [-0.029185979524046838, 0.6279504926791785, 0.47200647074257274, 0.11575396576695024, -0.22891716290843026, -0.33372344159502537, 0.3573428859266595, 0.07882476754414154], [0.14404302101748323, -0.5233757960922578, 0.27823154446745424, 0.13632610146430887, -0.33312920532434337, -0.1414574422349847, 0.07911516130695587, 0.015740039506873086]], "dtype": "float64"}, 2, 2], "e": {"__tup__": [{"__nd__": [[[-0.09165001330604414, -0.7083245850341584, -0.7661643692767074, -0.29525093425810567, 0.9069661229266281, -0.2916296650670056, -0.36947300438125225, 0.5921389527873567], [-0.09315148514265256, -0.7729696529118862, -0.8609347938187146, -0.36677208240292836, 0.9861555465793834, -0.2849358688634559, -0.40788021830061805, 0.6021111867215502], [-0.06988195334093858, -0.6588750092110371, -0.7170461360468761, -0.29608161964899804, 0.8342850534594607, -0.2548039236374715, -0.34195025187173456, 0.5221413561421707], [-0.08550053844244884, -0.5959891027532297, -0.6512367386996336, -0.26021816381056473, 0.7617876803121265, -0.23951669705217984, -0.31062220678311625, 0.49090165995265744]], [[0.006596880273355241, -0.2740391397354028, -0.19972415877255972, -0.08503134744410346, 0.2178767377511139, -0.06386433988312724, -0.08128172090127894, 0.12412968716434213], [-0.1455220408945454, 0.1735928112752085, 0.13405227042256965, 0.05736728598063467, -0.1891641370754962, 0.06597936279459445, 0.08229549537089396, -0.0887326994416351], [-0.14655337847412755, 0.04337502023445397, -0.024666699036148605, -0.0126465995040969, -0.008222868112182265, 0.023427382076431313, 0.005034614027958316, 0.009300918961637886], [-0.20107034103748359, 0.27978246396128614, 0.14595792354262455, 0.057113017039703624, -0.19555605399570294, 0.06227440869083656, 0.08402471888860319, -0.07672107400025768]]], "dtype": "float64"}, {"__nd__": [[[-0.2718252314466112, 0.5107565586582697, 1.1561909653925087, 0.7315475927560489], [-0.784219986236127, 0.4478584965112016, 0.8741366041481966, 0.758233026614171], [0.01085753468830896, 0.37613833516642176, 0.794841050407737, 0.5110435799613143], [-0.11321164384026894, 0.2926329742854744, 0.7985158060472795, 0.3229930544700304]], [[-0.5491066433649423, -0.15576281085882512, 0.24003945263460605, 0.12236256348036047], [0.4870259446362982, 0.17767374261939417, -0.25322836791941117, -0.5825219281062342], [-0.07011594687984504, 0.4257784146736402, 0.46717322764584696, 0.3410919146640998], [0.2793592921967107, 0.05867349374499174, -0.43241690376394987, -0.8071470048618751]]], "dtype": "float64"}, {"__nd__": [[[1.3654699459660282, -0.5068149049219249], [-0.4611364130085438, 0.8909715774226679], [-0.3470863574800285, 0.4396758848330613], [-0.38009114188995496, -0.8791045786350693]], [[0.35874690294726685, -1.1268721615962831], [-0.35432425965059344, -0.4456773529685931], [-0.16494810108651223, 0.693218352496299], [-0.17132515693354142, 0.4921003355198952]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-0.292171185803555, -0.10348826806596086, -0.25197737820688537, 0.15256251210246857, 1.471491972993157, -2.5666584409312976], [-0.23685026450968424, 0.17651242137244696, 0.2959939896870995, -0.37191458132128985, -1.7567217824785826, 0.32799548371410964], [1.727350214164185, -1.5338614049161376, 0.8638280136981883, -0.3285252231228939, -0.0613243458288473, -1.052898510703949], [-0.33445617235587666, 1.3000445946989585, 0.582655241987707, 1.732311605409944, 1.1774118889776937, 0.4390866789295762], [1.743934526167792, 0.43899315873829875, 0.8279881767970589, -0.29657095353470836, 0.06654581585920483, -0.6974238233002769], [0.989583934594853, -1.1783036231148656, 0.7823503424176613, -0.19065105750515945, 1.1712470934673194, 0.7508689887312598]]], "dtype": "float64"}, {"__nd__": [[0.5461938472940404, 0.21923240734159344, -0.47161207669597216], [-0.02008595186946025, -0.3516021577784448, -0.15548395270988238], [0.45336852998301647, 0.1912601432693088, -0.2096791285736239], [-0.30411520993211966, 0.009834627862878238, -0.3649680448143383], [-0.20134208321897432, 0.09360284146537316, 0.34659361083200885], [0.18262844166733191, -0.68738685257943, 0.09131001870907794]], "dtype": "float64"}, {"__nd__": [[0.021610071969773374, 0.12416708561180792, 0.4848629044953622, -0.6189714830414228, -0.17733102755733168, 0.1772718968380064], [-0.47447831983618266, 0.44278471441605266, 0.11050698381019751, 0.2539751957530212, -0.17128309845653836, 0.24412910698986315], [0.32054146679671147, 0.0698634060411505, 0.07032026780960877, 0.08110297178451777, -0.2590035794218145, -0.04425860407210588]], "dtype": "float64"}, {"__nd__": [[-0.04575675973570764, 0.11501815930604853, 0.29994727407308835, -0.3175608245935228, -0.03750270909587329, 0.44443666429769196], [-0.22307646864376923, -0.2466750051870961, 0.060691857549358075, 0.253315557145596, 0.003427816668197799, 0.3986881771310968], [0.25703819476995804, 0.25254602006701854, 0.166234950397228, 0.6982959301038255, -0.061548506659813255, -0.6010566876199541]], "dtype": "float64"}, {"__nd__": [[0.48127630845514224, -0.13730982928704769, 0.03236413330290991, 0.39286520545815806], [-0.48067786231767756, -0.37549416423184184, -0.4803833811356561, -0.23824088700762713], [0.13189098180509806, 0.1572563528438313, 0.08288225379856612, -0.4238297651677099], [-0.6930310309897932, 0.01630607561711468, -0.14153281009150073, 0.13781573151498352], [0.21058608786638103, 0.04147242961760458, 0.22803992561275613, 0.06876341223452052], [0.15901941168044, -0.2114019788609647, -0.05388342414943756, 0.05903282899708717]], "dtype": "float64"}, {"__nd__": [[0.24615854262522563, -0.11812235166737163, 0.15635017671963378, -0.0797516375746137, -0.035262650198439105, 0.2488557126072219, -0.5979181113162467, -0.38894169842242826, -0.4446556192328462, -0.7000848359544914, -0.20347933204655302, 0.22483016991832214, -0.08546521991477242, 0.05933702458255564, 0.3267652490132378, 0.39830583968030747, -0.020741380417841866, 0.4060757668708192], [0.02763799753023276, -0.2512194671482386, -0.17832010565962056, -0.4441609537695049, -0.2664401561200957, -0.10740500642232874, 0.24107550581358048, 0.5162309493497951, -0.414654454613112, 0.11784824047797568, -0.3121631817472603, 0.1424091265389606, -0.03932599520631798, -0.5492717477542591, 0.2784890974705453, -0.1815002138426193, -0.1601700715120664, -0.32092572338690695], [-0.19628498300626665, 0.12836713220847515, -0.05677330228092165, 0.09859860068474431, 0.1085765561786531, 0.3961984966584501, -0.10283584525931379, -0.4430573450537195, 0.3201667249715949, -0.0994464516291764, 0.3343777333732131, 0.11501313395474112, -0.039341259061003477, 0.10463276821388853, 0.5853037680378899, 0.623094158716126, 0.02081434053983361, 0.04805717799512259], [0.32287204723991564, -0.25369830983020414, 0.09992111178547654, -0.00775885438615692, 0.09417246343726608, -0.25001064178482174, -0.4768702479926907, -0.6218950307973875, -0.33521523389688235, -0.1376025854817999, -0.08795747599462854, 0.5811693487288551, 0.3317980109721894, -0.28862733490248815, 0.10431253573528708, -0.12212347090509752, -0.08530915141202854, 0.0555976948246146], [0.185751335092618, -0.10177754516520461, 0.3191554598203075, -0.3425814678427212, 0.0019017187086805326, 0.7793021179787497, 0.06692392278758252, 0.42996435198185184, 0.02745605345196245, 0.174233128598939, -0.01703495831780746, -0.05112227434926133, -0.23384471901762588, 0.12909040768726596, -0.2554611567557304, 0.19967557089402874, 0.32558610133377297, 0.10995942217116898], [-0.08587462006670755, 0.13618967379260882, -0.09260191666392378, 0.2806641376395448, -0.5494218252645371, -0.10068221043559386, -0.5972435985371993, -0.4485182490681615, 0.4091586689441728, 0.2685554945915058, -0.21584406998543712, -0.4507510368122691, -0.8893586513524954, -0.16304865237979035, 0.7261245036742207, 0.13046528143909422, -0.16787168581484685, 0.13952406285009186]], "dtype": "float64"}, {"__nd__": [[-0.46828750589833285, -0.08919700880929063, 0.029843241904721547, -0.025830195546314553, 0.23724183650702416, 0.1033935678708157], [0.2004978054323991, -0.2065116846692278, 0.26934462252316443, 0.4886810842871974, -0.2910448558954238, -0.2663086967143679], [0.40073530089606985, -0.05740319600851808, 0.4211464176199671, -0.13276071356765515, 0.4365136728812134, 0.03944574504163565], [0.07746864699680855, 0.4694154065009713, -0.10853114323325137, -0.2823366287774875, -0.1345692624085063, 0.1357001851929416], [-0.4697277216541543, 0.19124127079533645, -0.16163139528532297, 0.34434379822006905, -0.7182780914701414, -0.2359697325312506], [-0.5059404453702305, -0.24786883990918576, 0.07429977033268294, -0.05376798763492647, -0.07601327068440449, -0.047755461414018235]], "dtype": "float64"}, 3, 4], "e": {"__tup__": [{"__nd__": [[[0.045827152481686395, 0.6518285998968562, -0.31754889521887575, -0.5074876053101124, -0.1480746100214844, 0.2387169844113418], [-0.06573763766246453, 0.0712185165208029, -0.07330869797198518, -0.1350153068238045, -0.017467453371499263, 0.045730294679496374], [-0.33334407629452517, -0.2527988079907308, 0.03276498763854645, -0.17418628835015593, 0.01996605993569698, -0.021538442827343355], [-0.2757362605759765, -0.09871761377998121, -0.021878690299647816, -0.09395193276830416, -0.057863133296217395, -0.007309735889310103], [-0.3783948155222651, -0.2474178490387468, 0.010845512347576008, -0.06853600552779138, -0.04792440546915314, -0.045315181634776186], [-0.24531529500867388, -0.23909692853743725, 0.05046917616634424, -0.05018646854062995, 0.011377804387519211, -0.0436585877076387]]], "dtype": "float64"}, {"__nd__": [[[-1.0831567282555505, 1.8276635692369037, 0.42668426596298475], [0.5279909911434028, -0.4509271242414858, -0.4209895001658911], [1.2858770366761052, 1.7979935321595686, -0.7547686934959417], [-0.628328140291561, -0.5935600633347881, -0.35063507440357955], [1.2685163339452323, 0.8690493391764144, -0.9967073211101116], [0.878153620914236, 0.3724950061716293, 0.09655519672750901]]], "dtype": "float64"}, {"__nd__": [[[-0.3281074264708578, 0.6454628894891271, 0.47164038833282507, -0.012640028619296265], [0.2460561041572527, -0.13133030571401214, -0.4192384007317764, -0.414579139652203], [-1.5869835136028247, 0.6778381899515665, 1.1761517171537699, 0.5799517278092011], [1.658760464022543, -0.3559186480465841, 0.35688112520154747, -0.3532783468228995], [-0.5463348742962255, -0.13471311801724917, -0.6461904124564976, 0.1468823173199311], [1.345219451750282, 0.29411536395649424, -1.316499180704444, 0.4517642157784018]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[0.20338824061994343, -1.0085360419431078, 0.7068496408990508, 0.6626659703854839]], [[0.385037937656101, 0.5565334427512632, 0.2964180008086595, 2.0350733027310675]]], "dtype": "float64"}, {"__nd__": [[-0.026128251315752967, -0.09212496550037302], [-0.22605827410721335, -0.30967880117335106], [-0.3733415362803226, -0.26663919396927555], [-0.021204114495621394, 0.10028853854931434]], "dtype": "float64"}, {"__nd__": [[0.015342617565648534, -0.22966058322891386, 0.270055369205989, 0.22182380169028426], [-0.0478944922266051, -0.1958748433994136, 0.16452837624893984, 0.056392076245831334]], "dtype": "float64"}, {"__nd__": [[-0.43443817782451427, -0.02039407679532147, 0.07861074362231431, -0.26990843594632613], [0.0569530178512329, -0.4364467455773367, 0.40085583000365127, 0.3743849955178295]], "dtype": "float64"}, {"__nd__": [[-0.07575520029088942, 0.10903630135172195], [-0.7229765897399625, -0.3469042980795998], [-0.08813367604563895, -0.32163990642805773], [0.21431894478919764, 0.5991889592243982]], "dtype": "float64"}, {"__nd__": [[-0.3529844158287906, -0.2512390212255578, 0.0706345104929791, 0.48333484454988623, -0.36671229376197095, 0.07471083669208259, 0.5463896552439326, -0.4955277444537802], [-0.3843207620537496, -0.12708198193825146, -0.1561765238566233, 0.24378038632609336, 0.07249791594625142, -0.5324886178926385, 0.15462312087024324, -0.17326166620669514], [0.3823341650823717, -0.18827626093107588, -0.1909845849306707, 0.1623394832413594, 0.22887794470108097, 0.13442980918086106, -0.5056791952154425, 0.16141033197847532], [-0.3102924155023197, 0.07058283347647905, -0.42712033061971805, 0.13389664426291908, -0.24197967129221176, -0.3847903950781565, 0.21414604094471984, 0.07249335650302538]], "dtype": "float64"}, {"__nd__": [[-0.18419304041786913, 0.43535365470631926, -0.13219572647092273, 0.009632302191280398], [0.08067403430358028, -0.18589978023858408, 0.1413408880199788, -0.16003570414941734], [-0.12349149666486275, 0.4087927919410023, -0.31217581575372993, -0.723834099278847], [0.48328109850356826, 0.7647983857821009, -0.12158077963479955, -0.5810514121860556]], "dtype": "float64"}, 2, 2], "e": {"__tup__": [{"__nd__": [[[0.01812322290184997, 0.11529233675326556, -0.044627163045517254, -0.08049878782243819]], [[0.010800047149141671, 0.05815562353692691, 0.0016527469499658712, 0.007509163282771764]]], "dtype": "float64"}, {"__nd__": [[[-0.05527383800985945, 0.17156908096829288]], [[-0.28968643667361277, -0.08276034552897196]]], "dtype": "float64"}, {"__nd__": [[[0.7934648455856459, 0.5417532699651114]], [[-0.021498921850654367, 0.9729728654517638]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[-0.31048397619594376, -0.2862229498406692, -0.18992384102225704, -1.1133880419218483, 0.5795611424764774, 0.5245073752860862, -1.4944056198193065, 0.6991967316948878, 2.052684981945891, 0.17196033243705985, -0.33732516206850605, -0.1420032144075213, 0.6152567669685612, -1.7306716055072182, 0.16439070297730224, -0.3904639494628101], [1.8478250129560454, -0.17417273380495787, 1.6678876111195198, -1.1037406977820072, 0.5872591668027922, 0.3194002637912288, -0.8690472478647824, 0.17739611463694044, 1.2125188368316004, -0.3237917042866007, -1.6919626491697632, -0.01756282647744591, -0.9024230947925126, -0.3423409386638824, -0.08158776968512098, -1.7056522160155172], [-1.615658339320678, 0.48206683552552526, -0.5227186961696804, -2.564744341820567, 0.7848440366081351, 0.27236975527423984, -0.713874824036484, -1.3168302110582415, 0.8358078963143372, 0.3493506225604296, 2.3826022826840734, 0.42018859872493736, 0.3877031412620148, -0.16692793010856513, 0.816775867200791, 0.6250852012481537], [1.251725008565195, -0.5213229186372847, -0.4354074729439623, -0.4791031709119801, 0.7908017211789892, 1.498374440296732, -0.45884049314930064, -0.42477372779309125, 0.3140772238697992, -0.24576150001290142, 0.95205365313989, -2.2517772906190467, -0.8267050469168946, -0.7824163339614458, -2.320356012386145, -0.9636384443688041], [-0.9151561158459262, -0.20110465451883033, 1.1129658916602636, -0.24506065119070236, -1.0308137202711538, -0.056954541213528984, 1.0491736214413494, -0.9759588161149049, -0.9105748839214486, 0.5585489592742234, -0.22153611573080054, 0.6474846907094743, -0.013646791795296761, 0.701663648089567, -1.0350781140406409, -0.01208465078826744]]], "dtype": "float64"}, {"__nd__": [[-0.06320700430894705, -0.3647672772312392, -0.4690433843606065, 0.20572471058239852, -0.10528571466539342, -0.30668408449419887], [-0.028853681076999673, 0.33840565720479077, -0.6842213536115769, -0.4489916069709979, -0.2768659325685803, 0.43835366001609866], [0.08477609487578895, 0.23019517213836804, -0.34204827573786917, -0.3358607625811579, 0.13434411501681312, 0.017482299280633964], [0.16462164565923795, -0.05630129727853564, 0.08344311793638809, 0.04743572547153275, 0.2333302167254776, 0.24210248458334116], [-0.48596159831753094, -0.6741805666022567, 0.3005236192954521, 0.35631752797858096, -0.3061869058540012, -0.5579506219636886], [0.029710446302160274, 0.2792514679051588, 0.5392783567302943, 0.15488927163043528, -0.11151500897570849, -0.2679391864754588], [0.0034353875011963247, -0.08977941116557692, -0.3045203717186448, 0.6146266952589859, 0.5355505206216393, 0.34081460506662853], [-0.2762551193427383, 0.2565057972891836, 0.19188792284937053, 0.13276368726121587, 0.3748995801131331, 0.19061134285838321], [0.22200427089130065, 0.19107188501166303, 0.10223742240063176, -0.535083338914255, 0.02508632322316717, -0.16685758765473754], [-0.38395228729200614, 0.5045449857902472, 0.5186986022657111, 0.4077661800886836, 0.07656402505676238, 0.405187536285797], [0.0036159541805476625, 0.06083918306933381, -0.32804141113859486, 0.119097391349652, 0.018115778527325127, -0.3907956350503393], [-0.015359133544979071, -0.02391886915704254, 0.539268349993392, 0.26826399855402283, 0.0034336317274398045, 0.07463619294032489], [0.013263713431259417, -0.06087419543866432, -0.32472815921648607, -0.04531556328207877, -0.223829476995754, -0.3750946616769439], [0.1533665443642604, 0.11737939797997048, -0.5360122534104234, -0.03680539124853816, 0.29871041576997703, 0.31776712166889626], [0.3077510528655716, 0.01167396207043794, -0.25351359839074533, -0.32510964820741817, 0.10338147142510724, 0.11378401030954889], [0.38619933348151964, 0.32999389510262506, -0.03966859695935163, -0.3732617636582608, -0.09573197910326935, 0.0651693584431574]], "dtype": "float64"}, {"__nd__": [[-0.060624973616821225, -0.17336823242329674, 0.07586544449096967, -0.15118732242318314, -0.18841982511761218, 0.09343585029103187, -0.12059777403876143, 0.07323131669350563, 0.0819612068556368, -0.34182870032747237, -0.14437233715962364, 0.4313330819749387, -0.3486238538202861, -0.6349963346911592, -0.5585535459727534, 0.008732983973000761], [0.009275183409809672, -0.03528325662753628, 0.36425694867855046, -0.8018503299308813, 0.11878016232924708, 0.468431460541072, -0.3383342294675142, -0.11394195353696852, -0.22586757516858627, -0.26830381382325863, -0.097878601506912, 0.4282455851053514, 0.551216002646256, -0.10078176751204432, 0.571518064219341, 0.010672943095387459], [0.5261073080103242, -0.02798959489604753, 0.03931734781384329, 0.10964234179285036, 0.9536561038102606, 0.2553856724534367, -0.21217935898543908, 0.29069936108275735, -0.10854976112530244, -0.14692505487036273, 0.27258040796763333, 0.009325676331345836, 0.08357277435354182, 0.0041955665595877385, 0.10097493798163042, 0.12748976923084476], [-0.5810900798116245, 0.19997180058286007, -0.2946058259490902, -0.43268974066782334, -0.01752337315760473, 0.02519430099433628, -0.20805184937168422, 0.2493095573003923, -0.4026525499112721, -0.12208312463176768, -0.1754622404799002, -0.01397618110180181, 0.08365928539417163, -0.3023678848870634, 0.21728480323686847, 0.01890071050040992], [-0.5675838759935731, -0.5875774321191093, -0.0036957878787933704, -0.06629019533590623, -0.03102603407239005, -0.008394736959065393, 0.06766795185305421, 0.2842948829544899, -0.33333147027526744, -0.3515914778702384, -0.32799715736621476, 0.08667500015061207, 0.37348421359393363, -0.12940183147185313, -0.7505099532478964, -0.5111978587831126], [-0.24991101683880346, -0.16731866624856442, -0.12251658817641792, 0.011574336894275119, -0.09352389924673414, 0.31480188316302216, -0.2028051917044106, -0.2586953878426629, 0.14366266155613847, -0.46069262097783575, 0.11690751159600066, 0.030765955117880057, -0.044232746975450224, 0.4764828321447787, -0.18666223358841538, 0.6180894511572235]], "dtype": "float64"}, {"__nd__": [[-0.06762943476842978, -0.38310493349106844, 0.020975874298447588, -0.32287356009563045, -0.22552675434405006, 0.11910999000464734, 0.16667462659503743, -0.18665031007627744, 0.29622154351392016, 0.34725232740634016, 0.4308905269689873, 0.15882402213771665, 0.4090286136257096, -0.5642395283922172, -0.09537196458853876, -0.2601015768970279], [0.035767758338217624, -0.1714347657590212, -0.049847352584919714, 0.5646524640612148, -0.050915938926113485, 0.12413769515706417, -0.06968079897727056, 0.022713986387050102, 0.0018050041308214377, 0.13449721979194784, 0.34959226066415305, 0.4942181992809988, 0.09288602457275005, 0.17686406581068062, -0.34525935275567005, -0.026363022724686172], [0.2820868394650767, 0.25979059152514594, 0.06348291983851506, 0.26591818787038574, 0.14723001269814565, 0.3600918774777965, 0.08680774708713333, -0.1067094946713104, 0.10075237707990709, -0.8791783127558327, 0.11486572091551484, -1.094523847564435, -0.5170390222216253, 0.13553058028229523, 0.14325880118003856, -0.3487286340419791], [-0.21363061256368163, 0.4111622965842396, -0.14520903947180283, 0.6728760937208955, -0.0005759559529038447, 0.12241085306438004, 0.48506231954958223, 0.03930813485802587, -0.30070319420724034, -0.032918235508287035, -0.01068322085702728, -0.4094224956604696, -0.076749621402247, -0.222657628199231, 0.2773073271562302, 0.010383562255576686], [-0.08483872474427101, -0.031854579186772876, 0.06693660552184984, 0.18504425990693393, -0.2999136686008144, -0.31247628978638153, 0.331403764620759, -0.12370106166141917, -0.4250526277790622, 0.13314381281493137, 0.13901094180980605, -0.4592145771770119, 0.06884451463908188, 0.22067492772851208, 0.11231593551187892, 0.18959444512841858], [-0.4212807294272247, 0.09931204649885868, -0.09078592784113397, -0.14483704761739344, 0.26116668282591793, 0.443782353202269, 0.5383110163801077, 0.39444236145177153, -0.032920254489055105, 0.10581604890790292, 0.2300468615352731, 0.036353384789838586, 0.03922925637346331, 0.2471259395628546, -0.017784793807779663, -0.21878608091209845]], "dtype": "float64"}, {"__nd__": [[-0.12434191975075755, 0.1901731133201227, 0.0008979873599867195, 0.10206299306050136], [0.20102377266065694, -0.11245243886928288, 0.2268744490889418, 0.11365278103751833], [-0.37044392607344356, 0.4326914373558733, -0.15022342312707485, -0.4965286029555551], [-0.3135131128758907, -0.3063135482034966, 0.01565199559726004, -0.08215515166387115], [-0.10104959029217786, 0.18591058964474366, 0.10196260835781114, 0.09481436119328587], [0.1229485371986078, 0.18484040040067148, -0.6323860025332328, -0.10933147567933836], [-0.6540630193468422, 0.010817978039338623, -0.001389891667303596, 0.31365968243690406], [0.35629031781633624, 0.0608325144757102, -0.15010832321906994, 0.14554833376372134], [-0.15837529119729404, -0.0004177954330046697, 0.2958408950882303, -0.16733140379682707], [0.24170359177275028, 0.20322041060769072, -0.28643754513924646, 0.2921682384652203], [0.20956990361145153, 0.030577766211797232, -0.22869703740829567, -0.257761805572671], [-0.16129877307090673, 0.162778338987588, -0.28668755022041936, 0.13125367458596096], [-0.37252665866351775, -0.06122057533165468, 0.03289432637220259, 0.733539043517675], [-0.4131946966561102, 0.44160436994704, 0.04495272419618401, 0.12335157540478894], [0.03550437401646135, 0.1334180454585699, -0.04610553130860532, 0.43621505443723746], [-0.13694134326741672, 0.3396703716537743, -0.19331013606390687, -0.018075553920386084]], "dtype": "float64"}, {"__nd__": [[-0.32158876568706574, 0.13650711610425598, 0.43353778707796015, -0.023206853946256222, -0.059065709611172856, -0.33438508892037805, -0.06878779606569242, -0.4778382779164583, -0.2738633960156482, 0.06803581383274063, 0.3957039593224188, 0.8427632583688509, -0.1759755255212702, 0.43059004699363196, 0.07312574412049437, -0.045374294023912756, 0.12977833551784565, 0.018574945103987755, 0.033118717683665554, -0.12249994884800075, -0.4194328979396552, -0.46308756075662444, 0.1959734649843681, -0.08301002702571401, -0.17882666989684998, 0.00255168801694607, 0.23847967587468824, 0.054109182907919454, -0.1968164809922902, 0.3678878866946849, 0.47375565089448957, 0.14836699659685598], [0.292099053403227, 0.3725879990441433, 0.3390297949213541, 0.1842297531337871, 0.17950411412246484, 0.15589817200395079, -0.32920130236411305, 0.21021694705787572, -0.4067441913379662, -0.23837983653751646, 0.3910695203424262, 0.2520660488082592, 0.4462194869057588, -0.08150231510618768, -0.34567465238303025, -0.07213115912110285, 0.03060938917217226, 0.023696614902002904, 0.34010414362701047, -0.10838112750103365, 0.10561034154087727, -0.2965881448303818, 0.1350951834131439, 0.0009325650556910234, -0.22494400148535523, -0.07074880865245584, -0.05525272395456649, -0.08104653880112885, 0.531402372680305, -0.029533800692363887, -0.07316883305320132, -0.6292352872830425], [-0.2682622055938432, -0.07892202733690586, -0.20575580285299283, 0.41451227801333906, -0.04948133261886559, 0.3865332747954757, 0.018511390215110408, 0.01116835092647299, -0.026638934313750282, 0.0011388754864377396, 0.5156385009149578, -0.6958832180253655, -0.6004637233316036, -0.16295124931683416, 0.004358364581270664, 0.2070313274292037, 0.14192615940824255, -0.11527377386417105, 0.3057131079725287, 0.30905862094581926, 0.05521734938146965, 0.28879727174605235, 0.08179357902963864, -0.1684449080348651, 0.2093450517000014, 0.03318393677360331, 0.00040195402634607874, 0.44188607165687227, -0.735261638562026, -0.4253052510478709, -0.35612112791231104, -0.10897819427064168], [-0.07638245092980904, -0.4521976943319142, -0.2955474096932479, -0.258252503557071, 0.7372272276227675, 0.5405225843241439, -0.12352482530539478, -0.10907498709203968, -0.34476119915648823, -0.57243912396758, -0.03506405592255721, -0.2993555949732077, -0.02545515949860293, -0.4800617547130432, -0.22859233749307734, 0.044588128590849256, 0.10986287933518123, 0.12524158762260434, -0.39614669524711116, 0.2564057190406685, -0.2400635089805551, 0.18985737438219194, -0.003191602985457944, -0.41291624972338437, -0.09485931137435638, 0.10960912261913028, 0.1838896141423797, -0.04229636923332558, 0.4595336671803158, 0.30225323129596765, -0.07699623491992584, 0.22503654364108333], [0.580138045556456, 0.588145380234908, -0.3683999642467492, -0.2779599333471452, 0.4454756360201553, -0.31748431525129117, -0.3967623399875037, -0.14585823920971405, 0.1260680128423757, -0.03071901198332891, -0.19516908612167763, -0.2022637385173527, -0.2137011193238126, -0.26385288859768274, 0.6844898631828185, 0.08925332135566648, 0.2660277163214216, -0.1467232404356771, -0.055787003155814655, -0.2140661944111065, -0.7955125004757605, -0.41340278305435985, -0.5431856039941184, -0.6749351333854822, -0.35860769228943085, 0.39748670381448925, -0.013330850311306326, 0.3871691286963522, 0.12329477307026258, 0.23476764771454775, -0.27026909859137255, 0.15718528625780123], [0.21861427022230376, -0.17299418767010788, 0.1678621111848499, 0.16985632757244987, -0.16486578633902485, -0.3368305296812985, -0.35523849956948167, 0.025951662576438536, 0.09810296063679444, -0.23775774913737432, 0.007975602397829592, 0.17116111084154562, 0.18545721439959026, 0.46556403603674656, 0.383094249659413, -0.3003874298269632, 0.5050758628614067, -0.15929295136671442, 0.31346184373632824, 0.020395987815326654, -0.12364481353709027, -0.5423090579324799, -0.05146935306490446, -0.46781497059168264, 0.2901897707528317, 0.4550377971417744, -0.23907329678900707, 0.09059122275578134, -0.21761745660239262, -0.1885454190410369, 0.23234841115961138, -0.011561478553023754], [0.5218944125793035, -0.1446697436108763, 0.29980089974234675, 0.029224023887334593, 0.23873340292326994, -0.13414729719898402, -0.014867651265474813, -0.03464966378968042, -0.25081713445373616, 0.1986244603745258, 0.19507470167350924, 0.17806640847167, 0.4610896961752934, 0.434803440370891, -0.11248340695474061, 0.11191569553025348, -0.1870802067774638, 0.007006594834366637, -0.17992483473565943, 0.4860210927408153, 0.10910944510288885, -0.06540873843132664, 0.35808557155691306, -0.6511846229386234, -0.604230778459587, 0.23875159656593217, 0.018140534435210416, 0.0751338900782282, -0.39791883099556463, -0.00697011325510986, 0.6010528231204025, 0.24725988593875886], [0.05091196354284363, -0.13171617303148128, -0.11837185556081814, -0.6387593667600052, 0.07729212627634817, 0.25364493254455617, -0.4726857981511845, -0.1984250676920448, -0.3472393896700266, -0.289605358442629, 0.016044488046570066, -0.6253206464965423, 0.1842811315071192, 0.22615652098010466, -0.07533077641107375, -0.7442125529953625, -0.2989255878007169, 0.36987058023080893, -0.8333982842185718, -0.10425702365805577, -0.3540002335401777, 0.24137109047545097, -0.2025341856430384, 0.12118620622068346, 0.16963812575072792, 0.5508677519190059, -0.06088258310261106, 0.1128133108867834, -0.4455166437382308, 0.3571240689881232, -0.2281526544214094, -0.16822624922632368], [-0.006674338454763221, -0.46886530360195483, 0.0686310086786814, 0.2902398818775032, 0.08788014704918747, -0.4766692692536175, -0.037142788298897875, 0.2226086708938472, -0.5633966297897723, -0.3224298041453142, 0.262963081224456, 0.08102863025918595, 0.12359147463802614, 0.5578484422869935, 0.14056583355186078, 0.1973664031299538, 0.8132236581654978, -0.06414136742714721, -0.23936482908554851, 0.1283389194741398, 0.13169730881666847, -0.15801461758386245, -0.30971910586645424, 0.6929533266270234, 0.09805304111640688, -0.2196856144206227, 0.27477909068387957, -0.03274354546840035, -0.07586443389534937, 0.16868186157434797, -0.387564680469507, 0.10578816117515172], [0.5738864423195257, 0.13985329865919321, -0.11286363113503933, -0.15213158412847258, -0.1231019920035328, -0.07618880341433432, -0.11334195982748159, 0.27489343232237, -0.519962730962695, 0.25860001512364017, -0.11390595754276833, 0.11308433570861166, -0.33809637306025914, 0.39858932005483083, 0.3736904880515619, 0.2878771577250221, -0.45442643273806177, 0.24873632131981582, 0.12145652629071266, -0.48158137413977026, -0.007289215131244647, 0.10936034670997875, 0.16682258214694523, 0.053178410084656715, 0.0873694250542139, 0.4420834176759287, 0.36781017297320395, -0.860116486134823, -0.09523156645130461, -0.04939530826318538, -0.5256295718180913, 0.02825491746969365], [0.37462733172315194, -0.3259703578771771, 0.10092028160696169, -0.27474704318893134, -0.20157360710390304, 0.4432180902510105, -0.16276734603354748, 0.1386637467031874, -0.5613654348182961, 0.5586952684184641, 0.18079573389194967, -0.0547499527950785, 0.17266395299996842, -0.4229458286843431, 0.3547592225257971, -0.09719543264892905, -0.06122085451453041, -0.1474966844896453, -0.1741954208236754, 0.20840118348852257, 0.07677205503126686, 0.19552444242091196, -0.0026384068456714355, 0.1539599433231543, -0.1256546231775115, 0.6689195524228645, -0.4703791249291496, 0.3782811970688953, 0.41037223521629124, -0.25564764178967814, 0.668782083166792, 0.2986470848835954], [0.29134170204687343, 0.008406656107763005, -0.06516784746560293, -0.06282264811188243, 0.11715917889585516, 0.4203558825101688, 0.054394265358425876, -0.27343308694118706, 0.24732811871549393, 0.27634522391843763, 0.1963978585343358, 0.22364967195749477, -0.08127843904734243, -0.27869675013917383, -0.26045802919223393, -0.4906389745373574, 0.07108458346378492, -0.025034634967978083, -0.13199631135891263, -0.886685647303521, -0.3741952409082507, 0.33625219542095003, -0.1993878000189393, 0.10817750329705242, -0.41720562899257685, 0.4316116447077693, -0.04320829801640292, 0.08701211894506056, 0.6977339375645054, 0.4549320301768211, -0.09196968638611733, -0.17796605546110775], [-0.09766350345585954, 0.1529694745429607, -0.03156724027526134, -0.1185732435010415, 0.44015437521271267, -0.03426296753335106, 0.6898767279111958, 0.027222769125651903, 0.5355955182674182, 0.10719582681318802, 0.28888817197483435, -0.21738331545531733, -0.2561277667762235, 0.13791178650449626, 0.29656597273959456, 0.01207695060013886, -0.3859180005251091, -0.4080775098867981, -0.42814092065587234, -0.5302866338818143, 0.03284560783412625, 0.3375131765124076, -0.0040247360774426305, -0.6781244272013557, -0.05722813810036461, -0.04883969898978213, 0.6287996250676318, -0.4371413738758669, -0.11720328077975871, 0.5117558041633692, 0.1349935024039644, 0.1689939220549274], [0.03437533110716346, 0.09898306159717936, 0.23328285908356078, -0.24284589566782988, 0.26341757367245994, -0.41190090506566174, 0.09851039184946143, 0.053622288985898145, -0.18488792645763238, -0.006805582597694021, -0.1867620457996659, -0.3215103980585233, -0.1060844224840933, -0.3311965906114984, 0.09683923922797442, -0.345025648245709, 0.2135931068973154, -0.354443720378646, -0.16988847467430188, -0.1873810224645537, 0.39754401866245165, 0.09908698368497755, -0.06352014076331827, 0.149615832320917, -0.6321577041989064, -0.012940076892272234, 0.5992542533414288, 0.03970395192080177, 0.02037395075080065, 0.16443851389265882, 0.2686815729943037, -0.5992400078669579], [-0.504697944537308, 0.06308738732584396, -0.2144629685692529, 0.39360757866597423, -0.09455231696556717, -0.5803592207139353, -0.38040417505319907, -0.8249028058891726, -0.07023766118487376, -0.056515238179315, -0.12733881293456933, 0.3236637117089216, 0.3595559509550081, -0.05111652700683336, -0.6404081002430567, 0.18518447414767752, 0.16660806855257262, 0.0928174710470835, 0.2944439247924347, -0.349448428938312, 0.6026266158555417, -0.2878283259425604, 0.22476687816025637, -0.5062043083240054, -0.27907940618101523, 0.4549105940176008, -0.07826994690025497, 0.2519767815398394, 0.07108661661577383, -0.031487446800051805, 0.4091550974798069, -0.0709976473955701], [-0.5342214322690412, -0.2409720275855115, 0.16535751861050826, 0.11865273190875243, -0.47011747507063145, -0.2173027558224603, -0.17451640476067398, -0.1096408782342675, -0.41277407658046544, -0.1885113324358436, -0.2185831455778582, 0.23145997792857578, 0.2427363932970731, -0.0437431374266448, -0.11761193876121945, 0.041977293611734016, -0.24497253223667248, -0.34404962465023226, -0.28206636267771507, -0.190443949191437, -0.3490421399376813, 0.04861786771886242, 0.21891470213511252, -0.4761294902394725, 0.007538613364810005, -0.06598073857990067, 0.10245260566104088, -0.003881630431076418, -0.43607978397948927, -0.3787167896571496, 0.008827072675491817, 0.3086342759236528]], "dtype": "float64"}, {"__nd__": [[-0.6642562113129008, 0.10371797394047398, 0.35688660158629293, -0.04590776286186393, 0.16506074880793223, -0.21092072426934064, -0.035809859783063044, -0.48627405549244546, -0.7033029925256621, 0.28282974036758257, -0.29549847831733256, -0.13665835350102915, -0.41593267598105793, -0.4229909109251627, -0.052151310028384905, -0.21887315824069395], [0.11339129939840953, 0.13576152569387756, 0.09625489389738215, 0.0715802372785331, 0.4733304614795658, -0.3202781307134658, -0.016480459860875325, 0.33980694615587664, 0.37936194280141095, -0.42652909480337187, -0.09370663249676098, 0.40743632876017816, 0.15242373155971958, 0.05857890944393182, -0.06530540803039701, -0.11514027771232363], [0.08646571892709726, -0.075712237495345, -0.012773586313333253, 0.02357961183451121, -0.20342145854804974, 0.1596388121644315, 0.8742733945246258, -0.5330959917978225, 0.09720375725699397, -0.22805702658567167, 0.17322077325577498, 0.013967241998358276, 0.22855643221273902, 0.35265197295263384, 0.0577263353304544, 0.33961568652425345], [-0.7032893681366031, 0.47713568937480694, 0.11251864556770202, 0.32335300672355455, -0.36596272345245645, -0.2492309644816007, 0.04378148824343224, -0.005111366801238905, 0.08492057640299687, 0.32535454827898985, -0.5295660945908547, 0.2921457215151927, 0.002484572987378105, -0.06113455833736432, -0.4413345703154671, 0.1509146005130595], [-0.7242323553093112, 0.1945182033716138, -0.4476317506418976, 0.01603737378524221, -0.03575065119837613, 0.3508614441881651, -0.05359655070416705, -0.2082415683436997, -0.1519808087913609, 0.38186483062639903, 0.16088909075426808, -0.11568899445694839, -0.07093392598935402, -0.47015312717563407, 0.1165422602554623, -0.5279447123086397], [0.22553893975063682, 0.03858647774737975, 0.7160855398721767, -0.3300406661172917, -0.03079862731218988, 0.5394863102769187, 0.3574168920874243, -0.20886361547396992, 0.06719022851203432, 0.00059980692452617, 0.08458676021824715, -0.20960413079544893, -0.026796120810220356, -0.4794157125830141, 0.040537035560424615, 0.13500620202654962], [-0.06822582379316519, -0.4845152671754363, 0.2965595588554387, 0.10935751139784068, -0.21780452931029304, -0.08418013405141918, -0.4502008208345516, -0.3076230383333713, -0.16134340390714183, 0.36852624368404535, -0.19185101726624268, 0.19206409779862763, -0.16149486085847392, 0.07438426825556245, -0.04241865064314065, 0.26191698002782304], [-0.16044340146556735, -0.47175847500073226, -0.4145847094090898, -0.15956324818282722, -0.3747179912034158, 0.12306360470386538, -0.13973044945742333, 0.7123819650079294, -0.5213143785411396, 0.06108995361294036, -0.3383859070517007, -0.14802040211480358, 0.04242584200216325, -0.697611412462286, 0.07104783503640824, 0.5708722386822503], [-0.13603148951057784, 0.07794235704165285, 0.30747650793761977, -0.04918605000809896, -0.17428805655944496, -0.6827209211165256, 0.192297215818492, -0.033950522918669165, -0.30380515541644854, 0.03797901308330807, -0.1751041070418544, -0.15852783933465767, 0.18188734683261246, -0.40151472339715333, 0.3141945858221275, -0.4193058489355211], [-0.19881042250657913, 0.5768838582199731, 0.33820642849272664, 0.029281197748632054, -0.28215906465036494, -0.15878169200996797, 0.02198259437570892, 0.011256969701879527, -0.027261278887788856, -0.0092497542252553, -0.06431253291153423, -0.43259362069200386, -0.15495322264824984, 0.27438721619605394, -0.4105002892653352, -0.05686665346178355], [-0.24592617863125318, 0.033405128819433395, -0.11658995432004544, 0.4432180994924007, 0.15197494625707614, -0.3935409497331634, 0.2013684034841096, -0.2323955568786189, -0.17309891141326006, 0.09414570603213654, -0.1475242259190404, 0.2678567448968866, -0.23235301845401862, -0.18568396577740895, -0.19691126806443265, -0.16969162450651168], [0.32975746666434386, 0.18863888808309093, 0.5129831862036213, 0.6802932611697764, -0.30921510957970194, 0.021125738198186667, -0.3528600949930299, -0.2332817783244801, -0.18902475307670083, -0.3171464622414823, -0.31556618186952407, -0.043285313556076455, -0.3650155255159221, 0.2526380877412898, -0.27532606122525427, 0.8428706910289177], [-0.13541533528931854, 0.021605292575115665, -0.3886023553398105, 0.5118275274382917, 0.33585020424087486, 0.08907476122083376, -0.4040407646338697, 0.06320006331738608, -0.3364293138051765, 0.4387065790420082, 0.2427584195559518, 0.028631711546320833, -0.7309126989741966, -0.29502479359770917, 0.2063283394799617, 0.36386510751694673], [0.33275094344763945, -0.4776039006724366, 0.09046569204387121, 0.30465456494487886, -0.49750066967467355, -0.002371354611970904, 0.08215271949253687, -0.09022937545402047, -0.5556485566203605, -0.0977094330221186, -0.0981916290765072, -0.044421721364326254, -0.16904142762752086, -0.1329036459724402, -0.01191188470409337, 0.13273638149091332], [-0.09132802885700016, -0.12289467851040056, 0.027706211476337422, -0.5762747777685544, 0.14311053816793742, -0.00016892913688764245, 0.011908057979268548, 0.2882202922783392, -0.5858446590144745, -0.5921501609576891, 0.08214115359705586, -0.002135056538554732, 0.05105308295757966, -0.13863812255173366, 0.19930610227197934, 0.279791652897336], [0.1451946881854581, -0.07824853752137494, -0.6323068162843208, 0.23077086241113842, 0.36651859986527785, 0.193367523595194, -0.26389272821025195, 0.20213831810032734, 0.25815891238843364, 0.4676448743397257, -0.2628329772683873, -0.06793637536324154, 0.22418161534303654, 0.017223021431447334, -0.3351982806153824, 0.1243848855080109]], "dtype": "float64"}, 4, 4], "e": {"__tup__": [{"__nd__": [[[-1.0074861932255448, -0.04328603339944381, 0.12195207485110449, -2.3455945774505293, 1.1219357666999683, -0.5009285752408987, 2.787897838688386, -1.1017020174497985, -0.13969675591138486, -0.6295878808164741, 0.47709892350248817, 0.21925236260282288, 1.8050274323308755, -1.5681019244798402, 1.7921832884038889, -3.6478026601063656], [-0.43828892676432374, -0.9219694718965724, -0.6889003155020712, -2.0431035570855496, 1.0786902959457714, -0.5913128053181861, 1.7781240944231866, -0.3292687233596199, -0.14876874063432416, -0.5034730275341721, 0.4560407505940498, 0.4362504735585845, 1.7824810741343406, -1.1988062345031527, 1.855450452236474, -2.8196320988152253], [-0.8260635779808541, 0.6713893277151342, 0.19177176613769203, -0.8373602881195558, 0.9096690312062525, -0.6055005868566012, 1.4700463978738603, -0.8845665617278696, -0.5680303157233306, -0.43444946884391944, -0.045503063810863276, -0.2539997544720102, 0.6174941115338912, -0.8611612538956519, 0.8458658929464067, -1.8241581117415324], [-0.8487548818919408, -0.17399009279236677, -0.09316092623254846, -1.6424790404795284, 0.8625343513179483, -0.7128698370904983, 1.8709164827102227, -0.9025586571420061, -0.39866020743787023, -0.3930085685579161, 0.19206385203693155, 0.1316246210740044, 1.3479404077397803, -1.1197028102135418, 1.478214959071243, -2.71803485989403], [-0.8047855348533175, 0.4133523371840804, 0.32530432659304614, -1.2239812634258322, 0.666549858852868, -0.09609696764275619, 1.2391539707198727, -0.7740268707813909, -0.36118574974438494, -0.37836409787085923, -0.029490983967852, -0.2667379894840033, 0.6624086210200992, -0.5541070371605791, 0.39568177364614154, -1.4512654012772912]]], "dtype": "float64"}, {"__nd__": [[[-0.6026875508129832, 0.19893203656170286, 2.3946199696104378, -1.440703445231417, -1.4897585327582503, -2.082138287229345], [-0.8406075314590862, -1.0533489995370815, 0.30374142415230837, -0.6764310684004268, -0.5646400172792402, -1.1210631688962445], [0.13559479650121992, 0.6932762154790391, 0.18358906257715282, -1.2052551130653977, -1.7984192347162737, -1.8908961491953398], [-1.4189659668481236, -1.2275452791050538, 0.6747684712914461, 1.2119034226218928, -1.169667794096725, -2.77352052145719], [0.24698290231152578, 1.0162602237372502, -0.17187464178528467, 0.8338787626455438, 0.887273623636821, 1.5372607805949108]]], "dtype": "float64"}, {"__nd__": [[[1.8468545357287538, -0.4489039747061958, 0.25662249422401806, -0.15613296406888555], [-0.46712032536483405, 0.8069231264389435, 0.4915139447450494, -1.3593067691882623], [-0.24311695440441633, 0.7419721866934932, 1.5372626901688777, 0.016057840920735984], [-1.4185631801269287, -0.6008084081886809, 0.24233965734126392, -2.3526115276719004], [0.4360927156947329, 0.5544072306042459, 1.4606470382453969, -0.4485202313040475]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[0.5786616338151667, 0.08796289633100199, 0.21958492717994416, -0.25870590168573016, 1.4782789357160593, -0.9006921837126766, -0.5873207627637789, 0.010985513527525621], [0.8789540277008926, -1.4973731700945367, -1.3102967497654392, -2.037728507734718, 0.03269522525327079, -1.1183058172065439, -0.3845501100179167, 0.661378189891708], [-0.6415427207828466, 0.23128202274336457, -0.29933790775172203, 0.6721143150964368, 0.4751280673480088, -0.69193369617461, -0.16865175340045263, 1.0437433977204944]], [[0.393376870254097, -1.3969620577511315, 0.6556180611713827, -1.0846106224362382, 1.412513069961006, 1.024098839841025, 0.2898854759184008, 0.1572328434361934], [-0.20903225602742, 0.6785239514000281, -0.37077816946206166, 0.23556987063989387, 0.8719343665194511, -0.5618065435255897, -0.6161927399906946, -1.194365480724459], [-1.0644727461261723, -1.8283953114415024, 0.7298287890155296, 1.8089145420827708, 1.4253289434703056, -1.6260844886265164, 0.7248546715562, 1.4221063499235564]]], "dtype": "float64"}, {"__nd__": [[-0.1370429942215992, 0.030434229195014838, -0.6023679152414011, 0.08002085039299245, -0.31906791883041524], [0.059890505373454354, -0.4683608766566641, 0.16285746734327877, 0.30735380262800693, -0.23495395851455028], [-0.34107677442313034, -0.6468069127649944, -0.0799240602225531, -0.17447818028448153, -0.4653058237044164], [0.1359056268936515, -0.13499111455939475, -0.3227207875935697, -0.16637172169213413, 0.32139136167888466], [0.18088108312656667, -0.03900781844954695, 0.18573150553945575, -0.34827358691602167, 0.00971103021810916], [0.04419669914594117, -0.14095095254961806, 0.13063750078722536, -0.23413003639333535, 0.15294621233283537], [0.16555763145938757, -0.8751479642091199, 0.3570917110530353, -0.9142897696058262, -0.18481470766683758], [0.3076044102773265, -0.0951860175757322, -0.4398557947408626, -0.3281211185150012, 0.3080717238961859]], "dtype": "float64"}, {"__nd__": [[-0.5764513591645914, 0.33869733028860516, -0.0165306297126868, -0.35771859913273, -0.17493452169080734, -0.3051673023304924, 0.2598675602575586, 0.2197283429288515], [-0.515748340162814, -0.12107763555580021, -0.2589556987032831, 0.38805299053781106, -0.4050297354379263, 0.48126873140550414, -0.009988421855081269, -0.34858032191402627], [-0.20318225520278335, -0.24048565223053517, -0.3823855206046518, 0.028448057919112698, -0.36837466520755296, -0.6541098626214567, -0.11390294125445191, -0.5408568589253647], [-0.2256482604202561, -0.27273086658100676, -0.26136482160774427, 0.6002889304770461, 0.016923470150871228, 0.07361130269859964, -0.5329194889026884, -0.19618894009107657], [0.06459078898245359, -0.3073586738739019, 0.2724188530553956, -0.360808975770309, -0.08590984075955269, -0.17581549199266758, 0.04929619781457479, -0.19214702155662777]], "dtype": "float64"}, {"__nd__": [[-0.0812680881212812, -0.07083902753339566, -0.2203163272173903, 0.14552816054913378, 0.07151651245122759, -0.02658418357595019, -0.12913750003858743, 0.23896320915599908], [-0.0707502731442544, -0.09200918634778116, 0.0314070089664378, 0.5089760048912552, -0.4021068624063612, 0.2309570679166064, -0.5679513483714315, 0.06518194746858719], [0.4725248482826192, 0.14660055734019214, 0.05118752356193784, -0.16813484064092266, 0.009314160146566271, -0.07561884462100689, -0.2357929823794066, -0.08696268464246094], [0.05951434818793186, -0.23016512055919208, -0.24188454620540523, -0.4080730312776523, 0.11721750782666882, 0.1635969785901181, 0.06949813327095523, 0.28955114781859975], [0.026360291124218255, 0.117603281209727, 0.09115122769126885, -0.02746200582546236, -0.1592465484388942, 0.6642457847375733, -0.13558529395605254, -0.1997633133361309]], "dtype": "float64"}, {"__nd__": [[0.13020294703503207, 0.07555632458850153], [-0.4214374642830982, 0.33680347636912156], [-0.028258372605249993, -0.33327932856471615], [0.3566352343061622, 0.18770267147884617], [-0.37936259758437424, 0.5331548520773173], [0.1976392105098578, -0.0030588346875138396], [0.4770656416865338, 0.04778933514889217], [-0.008424922531209593, -0.1980900348065967]], "dtype": "float64"}, {"__nd__": [[0.17455513179212356, 0.14298089019719395, 0.004404041966913386, -0.23699590616028088, 0.03897895624982216, -0.020939635728641364, -0.40004229562026034, -0.4415437053972557, 0.16954601364293084, -0.7805664945155066, 0.06162315808837799, -0.14342491732614734], [0.39247398206397305, -0.14804046274616273, 0.19784526427556878, -0.11907415237133748, 0.034124446045843033, 0.22105095927601534, -0.13876671489737405, 0.2989664653495212, -0.2981093631319449, 0.08837785276751373, 0.27679191683635584, 0.19801124753710045], [0.5749334630206231, -0.22903179519319813, -0.4137447754045764, -0.29830053254853583, 0.1981327699645311, 0.14937688028081358, 0.03067699789422473, -0.33688711473213895, 0.23006018799871822, -0.3674114332951621, 0.3476848747130699, 0.8572315269419595], [0.10993534397146514, 0.17493244069732292, -0.14785793066866781, -0.5430639963260449, -0.1808158089090667, -0.3278685723296248, -0.4341013729245047, 0.5167079825986948, 0.0818587554077319, 0.11390627418419509, -0.09270670955887486, -0.05186132473303091], [0.2893652046718, -0.00970113560856463, 0.15920818140741982, -0.05880128935350475, -0.10264586721961415, -0.42207424968055857, -0.5933247187101384, -0.07846857031992054, 0.016949007597969808, -0.38583901101263723, -0.010342459510389864, 0.016125768897150606], [-0.6468024018178137, -0.27453739784385034, 0.12217916812064054, 0.11611761881872315, -0.3461024749199282, 0.04753441514992048, 0.30672216760814164, 0.23312188769726946, -0.028585736104180366, -0.14039289399728772, -0.1888033453003513, 0.14034593487289712], [0.058531191942866406, 0.18882120183066012, 0.03956214393634559, 0.24562918510423254, 0.028377759887310047, 0.23540661483950592, 0.26124683110999164, -0.24135798906744405, -0.5424432140612238, 0.463047856804366, 0.07699761921367851, -0.5143457686488364], [-0.21414584916561333, 0.2315622877109485, -0.08390304933986789, -0.15361842579573812, 0.42026455809389, -0.31418614391514316, -0.23944600732645016, 0.06341594251448027, 0.11713647961616468, 0.14890291030221944, 0.27337141337068405, 0.059573586204455055]], "dtype": "float64"}, {"__nd__": [[0.14681518168152274, -0.07776277779452496, 0.01880949097101427, 0.032260254016090326, 0.06452944074571082, 0.23176194227482821, 0.1665437045834493, 0.01326510087005493], [-0.24273779548231195, 0.020301377418812862, -0.00840533546872318, 0.05795447394395951, 0.6619575921421206, -0.15713183356237062, 0.12668670573760785, 0.32053862017328943], [0.029400574229814864, 0.7013535357516495, -0.113411787519443, 0.09722676669601411, -0.23964172346789459, -0.10545742054298794, 0.07468017311899737, 0.2538912186923189], [0.09572845020707225, 0.3777344548004208, -0.02848959317780152, -0.0996505589517016, 0.8111955321838159, 0.11827129483260923, -0.21584446379356428, -0.20827912867442064], [-0.28280397435940213, 0.28097080675703634, -0.10931318786797255, 0.038208742696945686, -0.019052602331480612, 0.21944772238228039, -0.25402322786520604, -0.3226716858429046], [0.06728919215447828, 0.1802877854099691, 0.09975969392929222, -0.24283611872919827, 0.20324545262741814, -0.22315827812869027, -0.156461593435915, -0.28318605741660063], [0.39034750459560924, 0.19002646230820475, 0.181498012703662, 0.3741634263427351, 0.843794774471617, -0.5127678311730466, -0.22151297426313243, -0.18485252525235715], [-0.25524588671480786, -0.3232749742811138, -0.08989591921660968, -0.3649811145868215, 0.10051271026122395, 0.01550949642858439, 0.14430859942909657, 0.15231853549147575]], "dtype": "float64"}, 2, 2, 500.0], "e": {"__tup__": [{"__nd__": [[[-0.037260608287276134, -0.12928142014017485, -0.030560975296947842, -0.1408579544706408, -0.06579953144592517, 0.07486853816380723, -0.009138632951962027, -0.02390491069750605], [-0.07736279113305457, -0.10597992478631227, -0.06931122823555856, -0.6320587750201645, -0.14888483902197697, 0.24068426591680098, 0.03315864606053118, -0.03621271681510557], [-0.07660331082922153, -0.08987583275856123, -0.03805595301794072, -0.45700593822275304, -0.1092823846060495, 0.12731568072044364, 0.03716025641769932, -0.007237629478333452]], [[0.0534736390411726, 0.43190871421677535, -0.06454133987119443, 0.23642716866906505, 0.3455792200656438, 0.0831928907408056, 0.024539403618545112, 0.20546924525664895], [-0.061534993943590965, 0.058225357363985195, -0.06236500370379919, -0.06929287758688085, -0.012494885462246086, 0.15105314589322008, 0.10169616157715472, 0.13455663870501203], [0.033519924023786354, 0.6415677744055142, -0.03853881586208263, 0.08514388134456613, 0.6703709780141109, -0.08153718454545991, -0.1115272295235166, 0.09542913172794325]]], "dtype": "float64"}, {"__nd__": [[[-0.05035945543898131, 0.4515425782165642, -0.3259627554881323, 0.3074796667018193, -0.4020915867294155], [0.05610625571603431, 2.2805807275864574, -0.5792213519293642, 0.5627700820449196, 0.13024739055753648], [0.6437108395281378, 0.10227652378484567, -0.2903371668423778, -0.23159404038785608, 0.7571529779966079]], [[-0.11147813670490621, -0.07949681413364085, 0.263653244331845, -1.380169792077238, -0.28572667141397773], [-0.10875842043484052, 0.5819878610095808, 0.5838917568238091, 1.0004598459183722, -0.17601859179627694], [0.7766879126438179, -0.4884138178723447, -0.613050053528424, -2.320475231846741, 1.0802886414476964]]], "dtype": "float64"}, {"__nd__": [[[-1.0792951002497613, 0.7122677158063757], [0.23292787762214237, -0.5852976289247346], [-0.12800580935993938, -0.4315011526962999]], [[0.038131097105285706, -0.13020500292749676], [-1.397393306313349, -0.2250622298498366], [-0.22837228598092643, 0.6963033635136386]]], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(mla, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")